# 07 — Iterative Backtest Notebook
Standalone experiment notebook. It rebuilds the same Silver/Gold feature
engineering as Pipe-1 from the raw source tables, trains the same two-stage
LightGBM model as `03_train_model.py`, and evaluates validation/internal
test/final inference in **iterative** mode:
- the scored horizon never sees its true `quantite`;
- week `t+1` uses the model prediction from week `t` to rebuild `lag_1`,
rolling stats, trends, zero-rates, expanding stats, etc.;
- validation and internal test are therefore block-forecast backtests.


%pip install lightgbm==4.3.0
dbutils.library.restartPython()


In [ ]:
import gc
import math
import sys
sys.path.append("./")

import numpy as np
import pandas as pd
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from mlflow.models import infer_signature
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, LongType, StringType, StructField, StructType
from pyspark.sql.window import Window



# -----------------------------------------------------------------------------
# Autonomous configuration: raw tables, splits, features, model params
# -----------------------------------------------------------------------------
NOM_EQUIPE = "telecacaton"
TABLE_PREDICTIONS = f"workspace.default.predictions_equipe_{NOM_EQUIPE}"

TBL_TRAIN = "workspace.default.histo_ventes_train"
TBL_TEST = "workspace.default.histo_ventes_test"
TBL_AGENCE = "workspace.default.donnees_agence"
TBL_ARTICLES = "workspace.default.donnees_articles"
TBL_FACTURATION = "workspace.default.donnees_facturation"

PAIR_KEYS = ["code_agence", "code_article"]

TRAIN_END_WEEK_ID = 202426
VAL_START_WEEK_ID = 202427
VAL_END_WEEK_ID = 202452
INTERNAL_TEST_START_WEEK_ID = 202501
INTERNAL_TEST_END_WEEK_ID = 202526
FINAL_INFERENCE_START_WEEK_ID = 202527
FINAL_INFERENCE_END_WEEK_ID = 202552

SEED = 42
OUTLIER_PERCENTILE = 0.995
ANOMALY_MULTIPLIER = 10.0
ANOMALY_ROLL_WINDOW = 26

LAGS_ALL = [1, 2, 4, 8, 13, 26, 52, 104]
ROLLING_WINDOWS = [4, 8, 13, 26, 52]
ROLLING_MEDIAN_WINDOWS = [4, 13]

FEATURES_NUMERIC = [
    "lag_1", "lag_2", "lag_4", "lag_8", "lag_13", "lag_26", "lag_52", "lag_104",
    "roll_mean_4", "roll_mean_8", "roll_mean_13", "roll_mean_26", "roll_mean_52",
    "roll_std_4", "roll_std_8", "roll_std_13", "roll_std_26", "roll_std_52",
    "roll_median_4", "roll_median_13",
    "zero_rate_26", "zero_rate_52", "pair_zero_rate_expanding",
    "recent_sum_4", "recent_sum_13", "recent_sum_26",
    "active_rate_4", "active_rate_13", "active_rate_26", "active_rate_52",
    "lag_1_is_zero", "lag_2_is_zero", "lag_4_is_zero", "lag_52_is_zero",
    "has_lag_1", "has_lag_13", "has_lag_52",
    "weeks_since_last_sale", "last_positive_qty", "nonzero_mean_expanding",
    "pair_active_rate_expanding",
    "trend_8", "ratio_n1_vs_mean", "yoy_ratio",
    "roll_mean_4_vs_13", "roll_mean_13_vs_52", "lag1_vs_roll13", "lag1_minus_roll13",
    "roll_std_13_ratio", "roll_std_26_ratio",
    "sem_mean_vs_pair_mean", "lag52_vs_sem_mean",
    "pair_mean", "pair_median", "pair_max", "pair_count", "pair_cv",
    "sem_mean", "sem_max", "sem_median",
    "agence_mean", "agence_median",
    "article_mean", "article_median",
    "n_active_weeks",
    "fac_prix_unit", "fac_pct_pro", "fac_nb_chantiers", "fac_nb_achats",
    "annee", "num_sem", "sin_sem", "cos_sem", "month_num", "quarter_num",
    "weeks_to_year_end", "weeks_from_year_start",
    "is_summer_trough", "is_xmas_trough", "is_q1", "is_q4",
]
FEATURES_CATEGORICAL = [
    "art_specialite_enc",
    "art_famille_enc",
    "art_marque_enc",
    "art_mdd_enc",
    "ag_region_enc",
]
FEATURES = FEATURES_NUMERIC + FEATURES_CATEGORICAL

LGB_PARAMS_ZERO = {
    "objective": "binary",
    "metric": "binary_logloss",
    "learning_rate": 0.03,
    "num_leaves": 63,
    "min_child_samples": 100,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,
    "reg_alpha": 0.3,
    "reg_lambda": 3.0,
    "scale_pos_weight": 1.5,
    "max_depth": 8,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_ZERO = 1200

LGB_PARAMS_QTY = {
    "objective": "tweedie",
    "tweedie_variance_power": 1.5,
    "metric": "None",
    "learning_rate": 0.02,
    "num_leaves": 127,
    "min_child_samples": 80,
    "feature_fraction": 0.75,
    "bagging_fraction": 0.75,
    "bagging_freq": 1,
    "reg_alpha": 0.3,
    "reg_lambda": 3.0,
    "max_depth": 10,
    "max_bin": 511,
    "n_jobs": -1,
    "seed": SEED,
    "verbose": -1,
}
LGB_NUM_ROUNDS_QTY = 1600

MLFLOW_EXPERIMENT = f"/Shared/sgdb2026_{NOM_EQUIPE}_iterative_notebook"


# -----------------------------------------------------------------------------
# Autonomous utility functions
# -----------------------------------------------------------------------------
EPS = 1e-10


def wape_numpy(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    return float(np.sum(np.abs(y_true - y_pred)) / (np.sum(np.abs(y_true)) + EPS))


def wape_lgb_feval(y_pred, dataset):
    y_true = dataset.get_label()
    return "wape", wape_numpy(y_true, y_pred), False


mlflow.set_experiment(MLFLOW_EXPERIMENT)



def materialize_table(df, table_name):
    """Serverless-friendly replacement for cache/persist."""
    if MATERIALIZE_INTERMEDIATE_DELTA:
        (
            df.write
            .format("delta")
            .mode("overwrite")
            .option("overwriteSchema", "true")
            .saveAsTable(table_name)
        )
        return spark.table(table_name)
    return df


## 0. Controls


In [ ]:
# Iterative validation early stopping controls.
# Checkpoints are dense at the beginning because the previous run peaked at the
# first checkpoints; later checkpoints are coarser to keep the overnight run sane.
RUN_ITERATIVE_EARLY_STOPPING = True
ZERO_THRESHOLD = 0.55
ZERO_THRESHOLD_CANDIDATES = [0.50, 0.55]
ZERO_ITER_CHECKPOINTS = [25, 50, 75, 100, 125, 150, 200, 300, 450, 600, 800, 1000, 1200]
QTY_ITER_CHECKPOINTS = [25, 50, 75, 100, 150, 200, 250, 300, 450, 600, 800, 1000, 1300, 1600]
ITERATIVE_EARLY_STOP_PATIENCE = 5
ITERATIVE_EARLY_STOP_MIN_DELTA = 1e-4
WARMUP_QTY_ROUNDS_FOR_ZERO = 200

# Train/validation mismatch reduction: create internal train folds, score them
# recursively with out-of-fold models, then add those degraded-feature rows to train.
RUN_OOF_TRAIN_AUGMENTATION = True
OOF_MAX_FOLDS = 3
OOF_FOLD_HORIZON_WEEKS = 26
OOF_MIN_HISTORY_WEEKS = 52
OOF_NUM_ROUNDS_ZERO = 150
OOF_NUM_ROUNDS_QTY = 200

# Stabilize recursive drift by blending the positive quantity prediction with a
# simple seasonal/recent-history baseline. This is tuned on blind iterative val.
RUN_BLEND_TUNING = True
PREDICTION_BLEND_ALPHA_GRID = [0.0, 0.10, 0.20, 0.35, 0.50]
MAX_DISPLAY_ROWS = 200

# Used only when RUN_ITERATIVE_EARLY_STOPPING = False.
MANUAL_ZERO_ITER = None
MANUAL_QTY_ITER = None

# Model artifact logging triggers noisy MLflow categorical validation in this
# notebook. Metrics and tables are still logged; flip this to True only if needed.
LOG_MLFLOW_MODELS = False

MATERIALIZE_INTERMEDIATE_DELTA = True
ITER_FEED_ROUNDED = True
TMP_PREFIX = f"workspace.default.tmp_iter_nb_{NOM_EQUIPE}"
TMP_SILVER = f"{TMP_PREFIX}_silver_ventes"
TMP_PANEL = f"{TMP_PREFIX}_panel"
TMP_TRAIN_FEATURES = f"{TMP_PREFIX}_train_features"

OUT_VAL_ITER = "workspace.default.iterative_nb_val_predictions"
OUT_INTERNAL_TEST_ITER = "workspace.default.iterative_nb_internal_test_predictions"
OUT_FINAL_ITER = f"workspace.default.iterative_nb_predictions_equipe_{NOM_EQUIPE}"

print(f"Train           : <= {TRAIN_END_WEEK_ID}")
print(f"Validation      : {VAL_START_WEEK_ID}..{VAL_END_WEEK_ID}")
print(f"Internal test   : {INTERNAL_TEST_START_WEEK_ID}..{INTERNAL_TEST_END_WEEK_ID}")
print(f"Final inference : {FINAL_INFERENCE_START_WEEK_ID}..{FINAL_INFERENCE_END_WEEK_ID}")
print(f"Iterative early stopping: {RUN_ITERATIVE_EARLY_STOPPING}")
print(f"Default zero threshold: {ZERO_THRESHOLD}; candidates={ZERO_THRESHOLD_CANDIDATES}")
print(f"OOF train augmentation: {RUN_OOF_TRAIN_AUGMENTATION}; folds={OOF_MAX_FOLDS}")
print(f"Blend tuning: {RUN_BLEND_TUNING}; grid={PREDICTION_BLEND_ALPHA_GRID}")


## 1. Pipe-1 Silver Reconstruction


In [ ]:
def _add_time_columns(df, semaine_col="semaine"):
    return (
        df.withColumn("annee", F.split(F.col(semaine_col), "-").getItem(0).cast("int"))
          .withColumn("num_sem", F.split(F.col(semaine_col), "-").getItem(1).cast("int"))
          .withColumn("week_id", F.col("annee") * F.lit(100) + F.col("num_sem"))
    )


def _encode_column(df, src, dst):
    if src not in df.columns:
        return df.withColumn(dst, F.lit(-1).cast("int"))
    labels = [
        row[src]
        for row in df.select(src).where(F.col(src).isNotNull()).distinct().orderBy(src).collect()
    ]
    if not labels:
        return df.withColumn(dst, F.lit(-1).cast("int"))

    mapping_entries = []
    for idx, label in enumerate(labels):
        mapping_entries.extend([F.lit(label), F.lit(idx)])
    mapping_expr = F.create_map(*mapping_entries)
    return df.withColumn(dst, F.coalesce(mapping_expr[F.col(src)], F.lit(-1)).cast("int"))


def build_silver_ventes(train_raw):
    raw = (
        train_raw
        .transform(_add_time_columns)
        .withColumnRenamed("quantite", "quantite_raw")
    )

    pair_stats = (
        raw.groupBy(*PAIR_KEYS)
        .agg(
            F.sum("quantite_raw").alias("_pair_sum"),
            F.expr("percentile_approx(quantite_raw, 0.995)").alias("_pair_p995"),
            F.expr("percentile_approx(quantite_raw, 0.5)").alias("_pair_median"),
        )
        .withColumn("is_dead_pair", (F.col("_pair_sum") == 0).cast("tinyint"))
    )

    capped = (
        raw.join(pair_stats, PAIR_KEYS, "left")
        .withColumn(
            "_cap_value",
            F.when(F.col("is_dead_pair") == 1, F.lit(None))
             .otherwise(F.greatest(F.col("_pair_p995"), F.col("_pair_median") * F.lit(2.0))),
        )
        .withColumn(
            "is_capped",
            (
                F.col("_cap_value").isNotNull()
                & (F.col("quantite_raw") > F.col("_cap_value"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite_capped",
            F.when(F.col("is_capped") == 1, F.col("_cap_value").cast("double"))
             .otherwise(F.col("quantite_raw").cast("double")),
        )
    )

    roll_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-ANOMALY_ROLL_WINDOW, -1)
    )
    with_roll = (
        capped
        .withColumn("_roll_median", F.expr("percentile_approx(quantite_capped, 0.5)").over(roll_w))
        .withColumn(
            "is_anomaly",
            (
                F.col("_roll_median").isNotNull()
                & (F.col("_roll_median") > F.lit(0))
                & (F.col("quantite_capped") > F.lit(ANOMALY_MULTIPLIER) * F.col("_roll_median"))
            ).cast("tinyint"),
        )
        .withColumn(
            "quantite",
            F.when(F.col("is_anomaly") == 1, F.col("_roll_median"))
             .otherwise(F.col("quantite_capped"))
             .cast("long"),
        )
    )

    smooth_w = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(-13, -1)
    )

    return (
        with_roll
        .withColumn("quantite_smooth", F.avg("quantite").over(smooth_w))
        .select(
            "semaine", "annee", "num_sem", "week_id",
            "code_agence", "code_article",
            F.col("quantite_raw").cast("long").alias("quantite_raw"),
            F.col("quantite").cast("long").alias("quantite"),
            F.col("quantite_smooth").cast("double").alias("quantite_smooth"),
            F.col("is_anomaly").cast("tinyint"),
            F.col("is_capped").cast("tinyint"),
            F.col("is_dead_pair").cast("tinyint"),
        )
    )


def build_articles_encoded(articles_raw):
    df = articles_raw
    mapping = [
        ("specialite", "art_specialite_enc"),
        ("famille", "art_famille_enc"),
        ("marque", "art_marque_enc"),
        ("article_mdd", "art_mdd_enc"),
    ]
    for src, dst in mapping:
        df = _encode_column(df, src, dst)

    keep = ["code_agence", "code_article"] + [dst for _, dst in mapping]
    return df.select(*keep).dropDuplicates(["code_agence", "code_article"])


def build_agences_encoded(agences_raw):
    src = "region" if "region" in agences_raw.columns else "ag_region"
    df = agences_raw.withColumnRenamed(src, "ag_region")
    df = _encode_column(df, "ag_region", "ag_region_enc")
    return df.select("code_agence", "ag_region_enc").dropDuplicates(["code_agence"])


def build_facturation_lagged(fac_raw):
    def _pick(*candidates, default=None):
        for c in candidates:
            if c in fac_raw.columns:
                return F.col(c)
        return F.lit(default)

    year_col = _pick("annee", "year")
    month_col = _pick("mois", "month")

    monthly = (
        fac_raw
        .withColumn("_annee", year_col.cast("int"))
        .withColumn("_mois", month_col.cast("int"))
        .groupBy("code_agence", "code_article", "_annee", "_mois")
        .agg(
            F.sum(_pick("sum_montant", default=0.0)).alias("_sum_montant"),
            F.sum(_pick("sum_quantite", default=0.0)).alias("_sum_quantite"),
            F.sum(_pick("nb_achats", default=0.0)).alias("fac_nb_achats"),
            F.sum(_pick("nb_achats_par_professionnels", default=0.0)).alias("_nb_pro"),
            F.sum(_pick("nb_chantiers", default=0.0)).alias("fac_nb_chantiers"),
        )
        .withColumn(
            "fac_prix_unit",
            F.when((F.col("_sum_quantite").isNull()) | (F.col("_sum_quantite") == 0), None)
             .otherwise(F.col("_sum_montant") / F.col("_sum_quantite")),
        )
        .withColumn(
            "fac_pct_pro",
            F.when((F.col("fac_nb_achats").isNull()) | (F.col("fac_nb_achats") == 0), None)
             .otherwise(F.col("_nb_pro") / F.col("fac_nb_achats")),
        )
    )

    return (
        monthly
        .withColumn("_shifted_mois", F.col("_mois") + F.lit(2))
        .withColumn(
            "_join_annee",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_annee") + F.lit(1))
             .otherwise(F.col("_annee")),
        )
        .withColumn(
            "_join_mois",
            F.when(F.col("_shifted_mois") > F.lit(12), F.col("_shifted_mois") - F.lit(12))
             .otherwise(F.col("_shifted_mois")),
        )
        .select(
            "code_agence", "code_article", "_join_annee", "_join_mois",
            "fac_prix_unit", "fac_pct_pro",
            F.col("fac_nb_chantiers").cast("double"),
            F.col("fac_nb_achats").cast("double"),
        )
    )


def build_silver_panel(cleaned, test_raw):
    test = (
        test_raw
        .transform(_add_time_columns)
        .withColumn("quantite_raw", F.lit(None).cast("long"))
        .withColumn("quantite", F.lit(None).cast("long"))
        .withColumn("quantite_smooth", F.lit(None).cast("double"))
        .withColumn("is_anomaly", F.lit(0).cast("tinyint"))
        .withColumn("is_capped", F.lit(0).cast("tinyint"))
        .withColumn("is_dead_pair", F.lit(0).cast("tinyint"))
        .select(*cleaned.columns)
    )

    pair_w = Window.partitionBy(*PAIR_KEYS)
    return cleaned.unionByName(test).withColumn(
        "is_dead_pair",
        F.max("is_dead_pair").over(pair_w).cast("tinyint"),
    )


In [ ]:
train_raw = spark.table(TBL_TRAIN)
test_raw = spark.table(TBL_TEST)
agences_raw = spark.table(TBL_AGENCE)
articles_raw = spark.table(TBL_ARTICLES)
fac_raw = spark.table(TBL_FACTURATION)

silver_ventes = materialize_table(build_silver_ventes(train_raw), TMP_SILVER)
articles_enc = build_articles_encoded(articles_raw)
agences_enc = build_agences_encoded(agences_raw)
fac_lagged = build_facturation_lagged(fac_raw)
panel = materialize_table(build_silver_panel(silver_ventes, test_raw), TMP_PANEL)

print(f"Silver panel rows: {panel.count():,}")


## 2. Pipe-1 Gold Feature Builder


In [ ]:
def build_features_from_y(panel_with_y, articles_enc, agences_enc, fac):
    df = panel_with_y
    pair_order = Window.partitionBy(*PAIR_KEYS).orderBy("week_id")

    for n in LAGS_ALL:
        df = df.withColumn(f"lag_{n}", F.lag("y", n).over(pair_order))

    def _lookback(n):
        return (
            Window.partitionBy(*PAIR_KEYS)
            .orderBy("week_id")
            .rowsBetween(-n, -1)
        )

    for n in ROLLING_WINDOWS:
        w = _lookback(n)
        df = (
            df.withColumn(f"roll_mean_{n}", F.avg("y").over(w))
              .withColumn(f"roll_std_{n}", F.stddev("y").over(w))
        )
    for n in ROLLING_MEDIAN_WINDOWS:
        df = df.withColumn(
            f"roll_median_{n}",
            F.expr("percentile_approx(y, 0.5)").over(_lookback(n)),
        )

    df = (
        df
        .withColumn(
            "_y_is_zero",
            F.when(F.col("y").isNull(), F.lit(None).cast("double"))
             .when(F.col("y") == 0, F.lit(1.0))
             .otherwise(F.lit(0.0)),
        )
        .withColumn(
            "_y_is_positive",
            F.when(F.col("y").isNull(), F.lit(None).cast("double"))
             .when(F.col("y") > 0, F.lit(1.0))
             .otherwise(F.lit(0.0)),
        )
    )
    df = (
        df
        .withColumn("zero_rate_26", F.avg("_y_is_zero").over(_lookback(26)))
        .withColumn("zero_rate_52", F.avg("_y_is_zero").over(_lookback(52)))
        .withColumn("active_rate_4", F.avg("_y_is_positive").over(_lookback(4)))
        .withColumn("active_rate_13", F.avg("_y_is_positive").over(_lookback(13)))
        .withColumn("active_rate_26", F.avg("_y_is_positive").over(_lookback(26)))
        .withColumn("active_rate_52", F.avg("_y_is_positive").over(_lookback(52)))
        .withColumn("recent_sum_4", F.sum("y").over(_lookback(4)))
        .withColumn("recent_sum_13", F.sum("y").over(_lookback(13)))
        .withColumn("recent_sum_26", F.sum("y").over(_lookback(26)))
        .withColumn(
            "pair_zero_rate_expanding",
            F.avg("_y_is_zero").over(
                Window.partitionBy(*PAIR_KEYS)
                .orderBy("week_id")
                .rowsBetween(Window.unboundedPreceding, -1)
            ),
        )
        .withColumn("lag_1_is_zero", F.when(F.col("lag_1").isNull(), None).otherwise((F.col("lag_1") == 0).cast("double")))
        .withColumn("lag_2_is_zero", F.when(F.col("lag_2").isNull(), None).otherwise((F.col("lag_2") == 0).cast("double")))
        .withColumn("lag_4_is_zero", F.when(F.col("lag_4").isNull(), None).otherwise((F.col("lag_4") == 0).cast("double")))
        .withColumn("lag_52_is_zero", F.when(F.col("lag_52").isNull(), None).otherwise((F.col("lag_52") == 0).cast("double")))
        .withColumn("has_lag_1", F.col("lag_1").isNotNull().cast("double"))
        .withColumn("has_lag_13", F.col("lag_13").isNotNull().cast("double"))
        .withColumn("has_lag_52", F.col("lag_52").isNotNull().cast("double"))
    )

    recent_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-4, -1)
    prev_w = Window.partitionBy(*PAIR_KEYS).orderBy("week_id").rowsBetween(-8, -5)
    df = (
        df
        .withColumn("_mean_recent4", F.avg("y").over(recent_w))
        .withColumn("_mean_prev4", F.avg("y").over(prev_w))
        .withColumn(
            "trend_8",
            F.when(F.col("_mean_prev4").isNull(), F.lit(None).cast("double"))
             .otherwise(
                 F.least(
                     F.greatest(
                         (F.col("_mean_recent4") - F.col("_mean_prev4"))
                         / (F.col("_mean_prev4") + F.lit(1.0)),
                         F.lit(-5.0),
                     ),
                     F.lit(5.0),
                 )
             ),
        )
        .withColumn(
            "yoy_ratio",
            F.when(
                F.col("lag_104").isNull() | (F.col("lag_104") == 0),
                F.lit(None).cast("double"),
            ).otherwise(F.col("lag_52") / F.col("lag_104")),
        )
    )

    df = df.withColumn("_pair_obs_idx", F.row_number().over(pair_order) - F.lit(1))
    pair_exp = (
        Window.partitionBy(*PAIR_KEYS)
        .orderBy("week_id")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("pair_mean", F.avg("y").over(pair_exp))
        .withColumn("pair_median", F.expr("percentile_approx(y, 0.5)").over(pair_exp))
        .withColumn("pair_max", F.max("y").over(pair_exp))
        .withColumn("pair_count", F.count("y").over(pair_exp))
        .withColumn("_pair_std", F.stddev("y").over(pair_exp))
        .withColumn("n_active_weeks", F.sum((F.col("y") > 0).cast("double")).over(pair_exp))
        .withColumn("last_positive_qty", F.last(F.when(F.col("y") > 0, F.col("y")), ignorenulls=True).over(pair_exp))
        .withColumn("_last_positive_obs_idx", F.max(F.when(F.col("y") > 0, F.col("_pair_obs_idx"))).over(pair_exp))
        .withColumn(
            "weeks_since_last_sale",
            F.when(F.col("_last_positive_obs_idx").isNull(), None)
             .otherwise(F.col("_pair_obs_idx") - F.col("_last_positive_obs_idx")),
        )
        .withColumn(
            "nonzero_mean_expanding",
            F.when((F.col("n_active_weeks").isNull()) | (F.col("n_active_weeks") == 0), None)
             .otherwise(F.sum(F.when(F.col("y") > 0, F.col("y")).otherwise(F.lit(0.0))).over(pair_exp) / F.col("n_active_weeks")),
        )
        .withColumn(
            "pair_active_rate_expanding",
            F.when((F.col("pair_count").isNull()) | (F.col("pair_count") == 0), None)
             .otherwise(F.col("n_active_weeks") / F.col("pair_count")),
        )
        .withColumn(
            "pair_cv",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("_pair_std") / (F.col("pair_mean") + F.lit(1e-6))),
        )
        .withColumn(
            "ratio_n1_vs_mean",
            F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None)
             .otherwise(F.col("lag_52") / F.col("pair_mean")),
        )
        .withColumn("roll_mean_4_vs_13", F.when((F.col("roll_mean_13").isNull()) | (F.col("roll_mean_13") == 0), None).otherwise(F.col("roll_mean_4") / F.col("roll_mean_13")))
        .withColumn("roll_mean_13_vs_52", F.when((F.col("roll_mean_52").isNull()) | (F.col("roll_mean_52") == 0), None).otherwise(F.col("roll_mean_13") / F.col("roll_mean_52")))
        .withColumn("lag1_vs_roll13", F.when((F.col("roll_mean_13").isNull()) | (F.col("roll_mean_13") == 0), None).otherwise(F.col("lag_1") / F.col("roll_mean_13")))
        .withColumn("lag1_minus_roll13", F.col("lag_1") - F.col("roll_mean_13"))
        .withColumn("roll_std_13_ratio", F.when((F.col("roll_mean_13").isNull()) | (F.col("roll_mean_13") == 0), None).otherwise(F.col("roll_std_13") / (F.col("roll_mean_13") + F.lit(1e-6))))
        .withColumn("roll_std_26_ratio", F.when((F.col("roll_mean_26").isNull()) | (F.col("roll_mean_26") == 0), None).otherwise(F.col("roll_std_26") / (F.col("roll_mean_26") + F.lit(1e-6))))
    )

    season_w = (
        Window.partitionBy(*PAIR_KEYS, "num_sem")
        .orderBy("annee")
        .rowsBetween(Window.unboundedPreceding, -1)
    )
    df = (
        df
        .withColumn("sem_mean", F.avg("y").over(season_w))
        .withColumn("sem_max", F.max("y").over(season_w))
        .withColumn("sem_median", F.expr("percentile_approx(y, 0.5)").over(season_w))
        .withColumn("sem_mean_vs_pair_mean", F.when((F.col("pair_mean").isNull()) | (F.col("pair_mean") == 0), None).otherwise(F.col("sem_mean") / F.col("pair_mean")))
        .withColumn("lag52_vs_sem_mean", F.when((F.col("sem_mean").isNull()) | (F.col("sem_mean") == 0), None).otherwise(F.col("lag_52") / F.col("sem_mean")))
    )

    ag_w = Window.partitionBy("code_agence").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    art_w = Window.partitionBy("code_article").orderBy("week_id").rowsBetween(Window.unboundedPreceding, -1)
    df = (
        df
        .withColumn("agence_mean", F.avg("y").over(ag_w))
        .withColumn("agence_median", F.expr("percentile_approx(y, 0.5)").over(ag_w))
        .withColumn("article_mean", F.avg("y").over(art_w))
        .withColumn("article_median", F.expr("percentile_approx(y, 0.5)").over(art_w))
    )

    two_pi = F.lit(2 * math.pi)
    df = (
        df
        .withColumn("sin_sem", F.sin(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("cos_sem", F.cos(two_pi * F.col("num_sem") / F.lit(52.0)))
        .withColumn("month_num", F.least(F.lit(12), F.greatest(F.lit(1), F.ceil(F.col("num_sem") / F.lit(4.333)))))
        .withColumn("quarter_num", F.ceil(F.col("month_num") / F.lit(3.0)))
        .withColumn("weeks_to_year_end", F.lit(52) - F.col("num_sem"))
        .withColumn("weeks_from_year_start", F.col("num_sem") - F.lit(1))
        .withColumn("is_summer_trough", ((F.col("num_sem") >= 30) & (F.col("num_sem") <= 35)).cast("tinyint"))
        .withColumn("is_xmas_trough", ((F.col("num_sem") >= 50) | (F.col("num_sem") == 1)).cast("tinyint"))
        .withColumn("is_q1", (F.col("quarter_num") == 1).cast("tinyint"))
        .withColumn("is_q4", (F.col("quarter_num") == 4).cast("tinyint"))
    )

    df = df.join(articles_enc, PAIR_KEYS, "left").join(agences_enc, "code_agence", "left")

    df = (
        df
        .withColumn(
            "_join_mois",
            F.least(F.lit(12), F.greatest(F.lit(1), F.ceil(F.col("num_sem") / F.lit(4.333)))),
        )
        .withColumn("_join_annee", F.col("annee"))
        .join(fac, ["code_agence", "code_article", "_join_annee", "_join_mois"], "left")
        .drop("_join_annee", "_join_mois")
    )

    base_cols = [
        "semaine", "week_id", "code_agence", "code_article",
        "quantite", "quantite_raw", "quantite_smooth",
        "is_anomaly", "is_capped", "is_dead_pair",
    ]
    for c in FEATURES:
        if c not in df.columns:
            df = df.withColumn(c, F.lit(None).cast("double"))

    return df.select(*base_cols, *FEATURES)


def panel_with_history_y(panel, history_end_week_id, pred_sdf=None):
    if pred_sdf is not None:
        base = panel.join(pred_sdf, ["semaine", "code_agence", "code_article"], "left")
    else:
        base = panel.withColumn("_iter_pred", F.lit(None).cast("double"))

    return base.withColumn(
        "y",
        F.when(F.col("_iter_pred").isNotNull(), F.col("_iter_pred"))
         .when(F.col("week_id") <= F.lit(history_end_week_id), F.col("quantite").cast("double"))
         .otherwise(F.lit(None).cast("double")),
    )


def build_static_features(history_end_week_id, start_week_id, end_week_id):
    return (
        build_features_from_y(
            panel_with_history_y(panel, history_end_week_id),
            articles_enc,
            agences_enc,
            fac_lagged,
        )
        .filter((F.col("week_id") >= F.lit(start_week_id)) & (F.col("week_id") <= F.lit(end_week_id)))
    )


## 3. Static Train Features for LightGBM Training
The model is trained only on the train period with complete real lag data.
Validation/test rows are built later by the iterative scorer, where lag_1,
lag_2 and the rolling features can consume previous predictions.


In [ ]:
train_features_sdf = materialize_table(build_static_features(TRAIN_END_WEEK_ID, 0, TRAIN_END_WEEK_ID), TMP_TRAIN_FEATURES)

cols_needed = ["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"] + FEATURES
train_pd = train_features_sdf.select(*cols_needed).toPandas()

print(f"Train rows: {len(train_pd):,}")


In [ ]:
# COMMAND ----------

# Build compact pandas state for the fast iterative scorer.
silver_pd = silver_ventes.toPandas()

test_pd = _add_time_columns(test_raw).toPandas()
test_pd["quantite_raw"] = np.nan
test_pd["quantite"] = np.nan
test_pd["quantite_smooth"] = np.nan
test_pd["is_anomaly"] = 0
test_pd["is_capped"] = 0
test_pd["is_dead_pair"] = 0

base_cols = [
    "semaine", "annee", "num_sem", "week_id", "code_agence", "code_article",
    "quantite_raw", "quantite", "quantite_smooth",
    "is_anomaly", "is_capped", "is_dead_pair",
]
panel_pd = pd.concat([silver_pd[base_cols], test_pd[base_cols]], ignore_index=True)
panel_pd = panel_pd.sort_values(["week_id", "code_agence", "code_article"]).reset_index(drop=True)

articles_pd = articles_enc.toPandas()
agences_pd = agences_enc.toPandas()
fac_pd = fac_lagged.toPandas()

for df in [panel_pd, articles_pd, agences_pd, fac_pd]:
    for c in ["code_agence", "code_article"]:
        if c in df.columns:
            df[c] = df[c].astype("int64")

print(f"Pandas panel rows: {len(panel_pd):,}")


def _mean(vals):
    return float(np.mean(vals)) if len(vals) else np.nan


def _median(vals):
    return float(np.median(vals)) if len(vals) else np.nan


def _max(vals):
    return float(np.max(vals)) if len(vals) else np.nan


def _std(vals):
    return float(np.std(vals, ddof=1)) if len(vals) > 1 else np.nan


def _zero_rate(vals):
    return float(np.mean(np.asarray(vals) == 0.0)) if len(vals) else np.nan


def _ratio(num, den):
    if pd.isna(num) or pd.isna(den) or den == 0:
        return np.nan
    return float(num / den)


def horizon_rows(start_week_id, end_week_id):
    cols = ["semaine", "annee", "num_sem", "week_id", "code_agence", "code_article", "quantite", "is_dead_pair"]
    out = panel_pd.loc[(panel_pd["week_id"] >= start_week_id) & (panel_pd["week_id"] <= end_week_id), cols].copy()
    out["_join_mois"] = np.ceil(out["num_sem"] / 4.333).clip(1, 12).astype("int64")
    out["_join_annee"] = out["annee"].astype("int64")
    return out.sort_values(["week_id", "code_agence", "code_article"]).reset_index(drop=True)


def prepare_state(history_end_week_id):
    hist = panel_pd.loc[(panel_pd["week_id"] <= history_end_week_id) & panel_pd["quantite"].notna()].copy()
    hist = hist.sort_values(["week_id", "code_agence", "code_article"])
    pair_hist, pair_sem_hist, agency_hist, article_hist, pair_sum = {}, {}, {}, {}, {}
    for row in hist.itertuples(index=False):
        ag = int(row.code_agence)
        art = int(row.code_article)
        sem = int(row.num_sem)
        pair = (ag, art)
        y = float(row.quantite)
        pair_hist.setdefault(pair, []).append(y)
        pair_sem_hist.setdefault((ag, art, sem), []).append(y)
        agency_hist.setdefault(ag, []).append(y)
        article_hist.setdefault(art, []).append(y)
        pair_sum[pair] = pair_sum.get(pair, 0.0) + y
    return {"pair": pair_hist, "pair_sem": pair_sem_hist, "agency": agency_hist, "article": article_hist, "pair_sum": pair_sum}


def clone_state(state):
    return {
        "pair": {k: v.copy() for k, v in state["pair"].items()},
        "pair_sem": {k: v.copy() for k, v in state["pair_sem"].items()},
        "agency": {k: v.copy() for k, v in state["agency"].items()},
        "article": {k: v.copy() for k, v in state["article"].items()},
        "pair_sum": state["pair_sum"].copy(),
    }


base_state_val = prepare_state(TRAIN_END_WEEK_ID)
base_state_test = prepare_state(VAL_END_WEEK_ID)
base_state_final = prepare_state(INTERNAL_TEST_END_WEEK_ID)
val_horizon = horizon_rows(VAL_START_WEEK_ID, VAL_END_WEEK_ID)
internal_test_horizon = horizon_rows(INTERNAL_TEST_START_WEEK_ID, INTERNAL_TEST_END_WEEK_ID)
final_horizon = horizon_rows(FINAL_INFERENCE_START_WEEK_ID, FINAL_INFERENCE_END_WEEK_ID)
print("Fast iterative states ready.")


## 4. Fast Pandas/Numpy Iterative Scorer

This scorer is defined before model training because iterative early stopping calls it after each LightGBM chunk. Each week is a pandas/numpy feature build over the horizon rows, then predictions are fed back into the in-memory state before the next week.


In [ ]:
def feature_rows_for_week(week_df, state):
    ag_stats = {ag: (_mean(vals), _median(vals)) for ag, vals in state["agency"].items()}
    art_stats = {art: (_mean(vals), _median(vals)) for art, vals in state["article"].items()}
    rows = []

    for r in week_df.itertuples(index=False):
        ag = int(r.code_agence)
        art = int(r.code_article)
        sem = int(r.num_sem)
        pair = (ag, art)
        vals = state["pair"].get(pair, [])
        sem_vals = state["pair_sem"].get((ag, art, sem), [])

        def lag(n):
            return vals[-n] if len(vals) >= n else np.nan

        def tail(n):
            return vals[-n:] if len(vals) else []

        lag_1 = lag(1)
        lag_2 = lag(2)
        lag_4 = lag(4)
        lag_13 = lag(13)
        lag_52 = lag(52)
        lag_104 = lag(104)
        pair_mean = _mean(vals)
        pair_std = _std(vals)
        recent4 = tail(4)
        prev4 = vals[-8:-4] if len(vals) >= 5 else []
        mean_recent4 = _mean(recent4)
        mean_prev4 = _mean(prev4)
        positive_vals = [v for v in vals if v > 0]
        n_active_weeks = float(len(positive_vals)) if len(vals) else 0.0
        pair_active_rate = n_active_weeks / len(vals) if len(vals) else np.nan
        last_positive_qty = float(positive_vals[-1]) if positive_vals else np.nan
        weeks_since_last_sale = np.nan
        if positive_vals:
            last_positive_idx = max(i for i, v in enumerate(vals) if v > 0)
            weeks_since_last_sale = float(len(vals) - last_positive_idx)
        nonzero_mean = float(np.mean(positive_vals)) if positive_vals else np.nan

        def active_rate(n):
            tv = tail(n)
            return float(np.mean(np.asarray(tv) > 0)) if len(tv) else np.nan

        def tail_sum(n):
            tv = tail(n)
            return float(np.sum(tv)) if len(tv) else np.nan

        trend_8 = np.nan
        if not pd.isna(mean_prev4):
            trend_8 = min(max((mean_recent4 - mean_prev4) / (mean_prev4 + 1.0), -5.0), 5.0)

        ag_mean, ag_median = ag_stats.get(ag, (np.nan, np.nan))
        art_mean, art_median = art_stats.get(art, (np.nan, np.nan))
        month_num = int(min(12, max(1, math.ceil(sem / 4.333))))
        quarter_num = int(math.ceil(month_num / 3.0))

        row = {
            "semaine": r.semaine, "week_id": int(r.week_id), "annee": int(r.annee), "num_sem": sem,
            "code_agence": ag, "code_article": art, "quantite": r.quantite,
            "is_dead_pair": 1 if state["pair_sum"].get(pair, 0.0) == 0 else 0,
            "lag_1": lag_1, "lag_2": lag_2, "lag_4": lag_4, "lag_8": lag(8), "lag_13": lag_13,
            "lag_26": lag(26), "lag_52": lag_52, "lag_104": lag_104,
            "zero_rate_26": _zero_rate(tail(26)), "zero_rate_52": _zero_rate(tail(52)),
            "active_rate_4": active_rate(4), "active_rate_13": active_rate(13),
            "active_rate_26": active_rate(26), "active_rate_52": active_rate(52),
            "recent_sum_4": tail_sum(4), "recent_sum_13": tail_sum(13), "recent_sum_26": tail_sum(26),
            "pair_zero_rate_expanding": _zero_rate(vals),
            "lag_1_is_zero": np.nan if pd.isna(lag_1) else float(lag_1 == 0),
            "lag_2_is_zero": np.nan if pd.isna(lag_2) else float(lag_2 == 0),
            "lag_4_is_zero": np.nan if pd.isna(lag_4) else float(lag_4 == 0),
            "lag_52_is_zero": np.nan if pd.isna(lag_52) else float(lag_52 == 0),
            "has_lag_1": float(not pd.isna(lag_1)), "has_lag_13": float(not pd.isna(lag_13)), "has_lag_52": float(not pd.isna(lag_52)),
            "weeks_since_last_sale": weeks_since_last_sale,
            "last_positive_qty": last_positive_qty,
            "nonzero_mean_expanding": nonzero_mean,
            "pair_active_rate_expanding": pair_active_rate,
            "trend_8": trend_8,
            "ratio_n1_vs_mean": _ratio(lag_52, pair_mean),
            "yoy_ratio": _ratio(lag_52, lag_104),
            "pair_mean": pair_mean, "pair_median": _median(vals), "pair_max": _max(vals),
            "pair_count": len(vals),
            "pair_cv": np.nan if pd.isna(pair_mean) or pair_mean == 0 else pair_std / (pair_mean + 1e-6),
            "sem_mean": _mean(sem_vals), "sem_max": _max(sem_vals), "sem_median": _median(sem_vals),
            "agence_mean": ag_mean, "agence_median": ag_median,
            "article_mean": art_mean, "article_median": art_median,
            "n_active_weeks": n_active_weeks,
            "annee": int(r.annee), "num_sem": sem,
            "sin_sem": math.sin(2 * math.pi * sem / 52.0),
            "cos_sem": math.cos(2 * math.pi * sem / 52.0),
            "month_num": month_num,
            "quarter_num": quarter_num,
            "weeks_to_year_end": 52 - sem,
            "weeks_from_year_start": sem - 1,
            "is_summer_trough": 1 if 30 <= sem <= 35 else 0,
            "is_xmas_trough": 1 if sem >= 50 or sem == 1 else 0,
            "is_q1": 1 if quarter_num == 1 else 0,
            "is_q4": 1 if quarter_num == 4 else 0,
        }
        for n in ROLLING_WINDOWS:
            tv = tail(n)
            row[f"roll_mean_{n}"] = _mean(tv)
            row[f"roll_std_{n}"] = _std(tv)
        for n in ROLLING_MEDIAN_WINDOWS:
            row[f"roll_median_{n}"] = _median(tail(n))
        row["roll_mean_4_vs_13"] = _ratio(row.get("roll_mean_4"), row.get("roll_mean_13"))
        row["roll_mean_13_vs_52"] = _ratio(row.get("roll_mean_13"), row.get("roll_mean_52"))
        row["lag1_vs_roll13"] = _ratio(lag_1, row.get("roll_mean_13"))
        row["lag1_minus_roll13"] = np.nan if pd.isna(lag_1) or pd.isna(row.get("roll_mean_13")) else lag_1 - row.get("roll_mean_13")
        row["roll_std_13_ratio"] = _ratio(row.get("roll_std_13"), row.get("roll_mean_13"))
        row["roll_std_26_ratio"] = _ratio(row.get("roll_std_26"), row.get("roll_mean_26"))
        row["sem_mean_vs_pair_mean"] = _ratio(row.get("sem_mean"), pair_mean)
        row["lag52_vs_sem_mean"] = _ratio(lag_52, row.get("sem_mean"))
        rows.append(row)

    feat = pd.DataFrame(rows)
    feat = feat.merge(articles_pd, on=["code_agence", "code_article"], how="left")
    feat = feat.merge(agences_pd, on="code_agence", how="left")

    fac_cols = ["code_agence", "code_article", "_join_annee", "_join_mois"]
    fac_join = week_df[fac_cols].merge(fac_pd, on=fac_cols, how="left")
    for c in ["fac_prix_unit", "fac_pct_pro", "fac_nb_chantiers", "fac_nb_achats"]:
        feat[c] = fac_join[c].values if c in fac_join.columns else np.nan

    for c in FEATURES:
        if c not in feat.columns:
            feat[c] = np.nan
    return feat


def append_predictions_to_state(state, scored_week):
    for r in scored_week.itertuples(index=False):
        ag = int(r.code_agence)
        art = int(r.code_article)
        sem = int(r.num_sem)
        pair = (ag, art)
        y_feed = float(r.prediction_int if ITER_FEED_ROUNDED else r.prediction)
        state["pair"].setdefault(pair, []).append(y_feed)
        state["pair_sem"].setdefault((ag, art, sem), []).append(y_feed)
        state["agency"].setdefault(ag, []).append(y_feed)
        state["article"].setdefault(art, []).append(y_feed)
        state["pair_sum"][pair] = state["pair_sum"].get(pair, 0.0) + y_feed


def baseline_quantity_from_features(feat):
    if "lag_52" in feat.columns:
        baseline = feat["lag_52"].astype(float).copy()
    else:
        baseline = pd.Series(np.nan, index=feat.index, dtype="float64")

    for c in ["sem_median", "sem_mean", "roll_mean_13", "roll_mean_26", "pair_median", "pair_mean", "article_median", "agence_median"]:
        if c in feat.columns:
            baseline = baseline.fillna(feat[c].astype(float))
    return baseline.fillna(0.0).clip(lower=0.0).values


def score_horizon_iterative_fast(base_state, horizon_df, zero_iter, qty_iter, threshold, label, zero_model=None, qty_model=None, return_features=False, blend_alpha=0.0):
    zero_booster = model_zero if zero_model is None else zero_model
    qty_booster = model_qty if qty_model is None else qty_model
    state = clone_state(base_state)
    out_parts = []
    for week_id in sorted(horizon_df["week_id"].unique()):
        week_df = horizon_df[horizon_df["week_id"] == week_id].copy()
        feat = feature_rows_for_week(week_df, state)
        X = feat[FEATURES].copy()
        for c in FEATURES_CATEGORICAL:
            if c in X.columns:
                X[c] = X[c].fillna(-1).astype("int32").astype("category")
        p_zero = zero_booster.predict(X, num_iteration=int(zero_iter))
        qty_raw = np.clip(qty_booster.predict(X, num_iteration=int(qty_iter)), 0.0, None)
        if blend_alpha > 0:
            baseline_qty = baseline_quantity_from_features(feat)
            qty = (1.0 - blend_alpha) * qty_raw + blend_alpha * baseline_qty
        else:
            qty = qty_raw
        pred = np.where(p_zero > threshold, 0.0, qty)
        pred = np.where(feat["is_dead_pair"].values == 1, 0.0, pred)
        if return_features:
            scored = feat.copy()
        else:
            scored = feat[["semaine", "week_id", "annee", "num_sem", "code_agence", "code_article", "quantite", "is_dead_pair"]].copy()
        scored["p_zero"] = p_zero
        scored["qty_pred"] = qty
        scored["prediction"] = pred
        scored["prediction_int"] = np.clip(np.round(pred), 0, None).astype(np.int64)
        scored["split"] = label
        out_parts.append(scored)
        append_predictions_to_state(state, scored)
    return pd.concat(out_parts, ignore_index=True)


## 5. Two-Stage LightGBM With Iterative Early Stopping


In [ ]:
def build_xy(df: pd.DataFrame):
    X = df[FEATURES].copy()
    for c in FEATURES_NUMERIC:
        if c in X.columns:
            X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    for c in FEATURES_CATEGORICAL:
        if c in X.columns:
            X[c] = X[c].fillna(-1).astype("int32").astype("category")
    y = df["quantite"].astype("float32").values
    is_zero = (y == 0).astype("int8")
    return X, y, is_zero


QTY_TRAIN_PARAMS = dict(LGB_PARAMS_QTY)
QTY_TRAIN_PARAMS["objective"] = "regression_l1"
QTY_TRAIN_PARAMS["metric"] = "None"


def _train_provisional_pair(train_slice_pd, fold_name):
    X_fold, y_fold, z_fold = build_xy(train_slice_pd)
    d_zero = lgb.Dataset(
        X_fold,
        label=z_fold,
        categorical_feature=FEATURES_CATEGORICAL,
        free_raw_data=False,
    )
    print(f"OOF {fold_name}: training provisional zero model on {len(train_slice_pd):,} rows")
    zero_model = lgb.train(
        LGB_PARAMS_ZERO,
        d_zero,
        num_boost_round=OOF_NUM_ROUNDS_ZERO,
        keep_training_booster=True,
    )

    nz = y_fold > 0
    X_fold_nz = X_fold.loc[nz].reset_index(drop=True)
    y_fold_nz = y_fold[nz]
    d_qty = lgb.Dataset(
        X_fold_nz,
        label=y_fold_nz,
        categorical_feature=FEATURES_CATEGORICAL,
        free_raw_data=False,
    )
    print(f"OOF {fold_name}: training provisional qty model on {len(X_fold_nz):,} non-zero rows")
    qty_model = lgb.train(
        QTY_TRAIN_PARAMS,
        d_qty,
        num_boost_round=OOF_NUM_ROUNDS_QTY,
        keep_training_booster=True,
    )
    return zero_model, qty_model


def _make_oof_train_folds(train_weeks):
    weeks = sorted(int(w) for w in train_weeks)
    folds = []
    end_idx = len(weeks) - 1
    while len(folds) < OOF_MAX_FOLDS:
        start_idx = end_idx - OOF_FOLD_HORIZON_WEEKS + 1
        hist_end_idx = start_idx - 1
        if start_idx < 0 or hist_end_idx < 0:
            break
        if hist_end_idx + 1 < OOF_MIN_HISTORY_WEEKS:
            break
        folds.append({
            "history_end_week_id": weeks[hist_end_idx],
            "start_week_id": weeks[start_idx],
            "end_week_id": weeks[end_idx],
        })
        end_idx = start_idx - 1
    return list(reversed(folds))


def build_oof_degraded_training_rows():
    train_weeks = train_pd["week_id"].drop_duplicates().sort_values().tolist()
    folds = _make_oof_train_folds(train_weeks)
    if not folds:
        print("OOF augmentation skipped: not enough historical weeks.")
        return pd.DataFrame(columns=train_pd.columns)

    parts = []
    for fold_idx, fold in enumerate(folds, start=1):
        fold_name = f"fold_{fold_idx}_{fold['start_week_id']}_{fold['end_week_id']}"
        history_end = fold["history_end_week_id"]
        train_slice = train_pd[train_pd["week_id"] <= history_end].copy()
        fold_zero, fold_qty = _train_provisional_pair(train_slice, fold_name)
        fold_horizon = horizon_rows(fold["start_week_id"], fold["end_week_id"])
        degraded = score_horizon_iterative_fast(
            prepare_state(history_end),
            fold_horizon,
            fold_zero.current_iteration(),
            fold_qty.current_iteration(),
            ZERO_THRESHOLD,
            f"oof_train_{fold_name}",
            zero_model=fold_zero,
            qty_model=fold_qty,
            return_features=True,
        )
        degraded["training_source"] = "oof_iterative"
        degraded["oof_fold"] = fold_name
        degraded_cols = ["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"] + FEATURES + ["training_source", "oof_fold"]
        degraded = degraded[degraded_cols].copy()
        parts.append(degraded)
        print(
            f"OOF {fold_name}: added {len(degraded):,} degraded train rows "
            f"from {fold['start_week_id']}..{fold['end_week_id']}"
        )
        del fold_zero, fold_qty, train_slice, fold_horizon
        gc.collect()

    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=train_pd.columns)


train_real_pd = train_pd.copy()
train_real_pd["training_source"] = "real_lags"
train_real_pd["oof_fold"] = ""

oof_train_pd = build_oof_degraded_training_rows() if RUN_OOF_TRAIN_AUGMENTATION else pd.DataFrame(columns=train_real_pd.columns)
if len(oof_train_pd):
    keep_cols = ["semaine", "code_agence", "code_article", "week_id", "quantite", "is_dead_pair"] + FEATURES + ["training_source", "oof_fold"]
    train_model_pd = pd.concat([train_real_pd[keep_cols], oof_train_pd[keep_cols]], ignore_index=True)
else:
    train_model_pd = train_real_pd

X_tr, y_tr, z_tr = build_xy(train_model_pd)
gc.collect()

print(f"Real train rows       : {len(train_pd):,}")
print(f"OOF degraded rows     : {len(oof_train_pd):,}")
print(f"Final model train rows: {len(train_model_pd):,}")
print(f"Zero rate final train : {z_tr.mean():.3f}")


In [ ]:
RUN_ID = None
early_stop_rows = []


def _iteration_checkpoints(max_rounds, checkpoints):
    vals = sorted(set(int(v) for v in checkpoints if 0 < int(v) <= int(max_rounds)))
    if not vals or vals[-1] != int(max_rounds):
        vals.append(int(max_rounds))
    return vals


def _train_lightgbm_to(params, dataset, previous_model, target_iter, label):
    before = 0 if previous_model is None else int(previous_model.current_iteration())
    if target_iter <= before:
        return previous_model
    print(f"Training {label}: rounds {before + 1}..{target_iter}")
    return lgb.train(
        params,
        dataset,
        num_boost_round=int(target_iter - before),
        init_model=previous_model,
        keep_training_booster=True,
    )


def _iterative_validation_wape(zero_booster, qty_booster, zero_iter, qty_iter, stage, threshold=ZERO_THRESHOLD, blend_alpha=0.0):
    pred_val = score_horizon_iterative_fast(
        base_state_val,
        val_horizon,
        zero_iter,
        qty_iter,
        threshold,
        f"validation_{stage}",
        zero_model=zero_booster,
        qty_model=qty_booster,
        blend_alpha=blend_alpha,
    )
    wape = wape_numpy(pred_val["quantite"].values, pred_val["prediction"].values)
    del pred_val
    gc.collect()
    return wape


def _restore_best_model(model_str, model_name):
    if model_str is None:
        raise RuntimeError(f"No best {model_name} model was captured during iterative early stopping.")
    return lgb.Booster(model_str=model_str)


with mlflow.start_run(run_name="iterative_early_stop_notebook") as run:
    RUN_ID = run.info.run_id
    mlflow.log_params({
        "n_features": len(FEATURES),
        "train_rows_real": len(train_pd),
        "train_rows_oof_degraded": len(oof_train_pd),
        "train_rows_final": len(train_model_pd),
        "train_end": TRAIN_END_WEEK_ID,
        "val_start": VAL_START_WEEK_ID,
        "val_end": VAL_END_WEEK_ID,
        "internal_test_start": INTERNAL_TEST_START_WEEK_ID,
        "internal_test_end": INTERNAL_TEST_END_WEEK_ID,
        "materialize_intermediate_delta": MATERIALIZE_INTERMEDIATE_DELTA,
        "iter_feed_rounded": ITER_FEED_ROUNDED,
        "iterative_early_stopping": RUN_ITERATIVE_EARLY_STOPPING,
        "zero_threshold_default": ZERO_THRESHOLD,
        "zero_threshold_candidates": str(ZERO_THRESHOLD_CANDIDATES),
        "oof_train_augmentation": RUN_OOF_TRAIN_AUGMENTATION,
        "oof_max_folds": OOF_MAX_FOLDS,
        "oof_fold_horizon_weeks": OOF_FOLD_HORIZON_WEEKS,
        "oof_num_rounds_zero": OOF_NUM_ROUNDS_ZERO,
        "oof_num_rounds_qty": OOF_NUM_ROUNDS_QTY,
        "zero_iter_checkpoints": str(ZERO_ITER_CHECKPOINTS),
        "qty_iter_checkpoints": str(QTY_ITER_CHECKPOINTS),
        "iterative_early_stop_patience": ITERATIVE_EARLY_STOP_PATIENCE,
        "iterative_early_stop_min_delta": ITERATIVE_EARLY_STOP_MIN_DELTA,
        "warmup_qty_rounds_for_zero": WARMUP_QTY_ROUNDS_FOR_ZERO,
        "log_mlflow_models": LOG_MLFLOW_MODELS,
        "blend_tuning": RUN_BLEND_TUNING,
    })

    dtrain_z = lgb.Dataset(
        X_tr,
        label=z_tr,
        categorical_feature=FEATURES_CATEGORICAL,
        free_raw_data=False,
    )

    nz = y_tr > 0
    X_tr_nz = X_tr.loc[nz].reset_index(drop=True)
    y_tr_nz = y_tr[nz]
    dtrain_q = lgb.Dataset(
        X_tr_nz,
        label=y_tr_nz,
        categorical_feature=FEATURES_CATEGORICAL,
        free_raw_data=False,
    )

    if RUN_ITERATIVE_EARLY_STOPPING:
        warmup_qty_rounds = min(WARMUP_QTY_ROUNDS_FOR_ZERO, LGB_NUM_ROUNDS_QTY)
        print(f"Training fixed warm-up qty model for zero early stopping: {warmup_qty_rounds} rounds")
        warmup_qty_model = lgb.train(
            QTY_TRAIN_PARAMS,
            dtrain_q,
            num_boost_round=warmup_qty_rounds,
            keep_training_booster=True,
        )

        model_zero = None
        best_zero_model_str = None
        best_zero_row = {"val_wape": np.inf}
        no_improve_zero = 0

        for zero_target in _iteration_checkpoints(LGB_NUM_ROUNDS_ZERO, ZERO_ITER_CHECKPOINTS):
            model_zero = _train_lightgbm_to(LGB_PARAMS_ZERO, dtrain_z, model_zero, zero_target, "zero")
            zero_iter = int(model_zero.current_iteration())
            checkpoint_rows = []
            for threshold in ZERO_THRESHOLD_CANDIDATES:
                val_wape = _iterative_validation_wape(
                    model_zero,
                    warmup_qty_model,
                    zero_iter,
                    warmup_qty_rounds,
                    f"zero_iter_{zero_iter}_thr_{threshold}",
                    threshold=float(threshold),
                )
                row = {
                    "model": "zero",
                    "checkpoint": zero_target,
                    "zero_iter": zero_iter,
                    "qty_iter": warmup_qty_rounds,
                    "threshold": float(threshold),
                    "blend_alpha": 0.0,
                    "val_wape": val_wape,
                }
                early_stop_rows.append(row)
                checkpoint_rows.append(row)
                print(f"zero zi={zero_iter:4d} fixed_qi={warmup_qty_rounds:4d} thr={threshold:.2f} -> iterative val WAPE={val_wape:.5f}")

            row = min(checkpoint_rows, key=lambda r: r["val_wape"])
            val_wape = row["val_wape"]
            if val_wape + ITERATIVE_EARLY_STOP_MIN_DELTA < best_zero_row["val_wape"]:
                best_zero_row = row.copy()
                best_zero_model_str = model_zero.model_to_string(num_iteration=zero_iter)
                no_improve_zero = 0
                print(f"new best zero model at {zero_iter} rounds; iterative val WAPE={val_wape:.5f}")
            else:
                no_improve_zero += 1
                print(f"zero no improvement {no_improve_zero}/{ITERATIVE_EARLY_STOP_PATIENCE}; best={best_zero_row['val_wape']:.5f}")

            if no_improve_zero >= ITERATIVE_EARLY_STOP_PATIENCE:
                print("Zero model iterative early stopping triggered.")
                break

        model_zero = _restore_best_model(best_zero_model_str, "zero")
        BEST_ZERO_ITER = int(best_zero_row["zero_iter"])

        model_qty = None
        best_qty_model_str = None
        best_qty_row = {"val_wape": np.inf}
        no_improve_qty = 0

        for qty_target in _iteration_checkpoints(LGB_NUM_ROUNDS_QTY, QTY_ITER_CHECKPOINTS):
            model_qty = _train_lightgbm_to(QTY_TRAIN_PARAMS, dtrain_q, model_qty, qty_target, "qty")
            qty_iter = int(model_qty.current_iteration())
            checkpoint_rows = []
            for threshold in ZERO_THRESHOLD_CANDIDATES:
                val_wape = _iterative_validation_wape(
                    model_zero,
                    model_qty,
                    BEST_ZERO_ITER,
                    qty_iter,
                    f"qty_iter_{qty_iter}_thr_{threshold}",
                    threshold=float(threshold),
                )
                row = {
                    "model": "qty",
                    "checkpoint": qty_target,
                    "zero_iter": BEST_ZERO_ITER,
                    "qty_iter": qty_iter,
                    "threshold": float(threshold),
                    "blend_alpha": 0.0,
                    "val_wape": val_wape,
                }
                early_stop_rows.append(row)
                checkpoint_rows.append(row)
                print(f"qty fixed_zi={BEST_ZERO_ITER:4d} qi={qty_iter:4d} thr={threshold:.2f} -> iterative val WAPE={val_wape:.5f}")

            row = min(checkpoint_rows, key=lambda r: r["val_wape"])
            val_wape = row["val_wape"]
            if val_wape + ITERATIVE_EARLY_STOP_MIN_DELTA < best_qty_row["val_wape"]:
                best_qty_row = row.copy()
                best_qty_model_str = model_qty.model_to_string(num_iteration=qty_iter)
                no_improve_qty = 0
                print(f"new best qty model at {qty_iter} rounds; iterative val WAPE={val_wape:.5f}")
            else:
                no_improve_qty += 1
                print(f"qty no improvement {no_improve_qty}/{ITERATIVE_EARLY_STOP_PATIENCE}; best={best_qty_row['val_wape']:.5f}")

            if no_improve_qty >= ITERATIVE_EARLY_STOP_PATIENCE:
                print("Qty model iterative early stopping triggered.")
                break

        model_qty = _restore_best_model(best_qty_model_str, "qty")
        BEST_QTY_ITER = int(best_qty_row["qty_iter"])
        BEST_THRESHOLD = float(best_qty_row["threshold"])
        BEST_VAL_WAPE = float(best_qty_row["val_wape"])
    else:
        model_zero = lgb.train(
            LGB_PARAMS_ZERO,
            dtrain_z,
            num_boost_round=LGB_NUM_ROUNDS_ZERO,
            keep_training_booster=True,
        )
        model_qty = lgb.train(
            QTY_TRAIN_PARAMS,
            dtrain_q,
            num_boost_round=LGB_NUM_ROUNDS_QTY,
            keep_training_booster=True,
        )
        BEST_ZERO_ITER = int(MANUAL_ZERO_ITER or model_zero.current_iteration())
        BEST_QTY_ITER = int(MANUAL_QTY_ITER or model_qty.current_iteration())
        BEST_THRESHOLD = float(ZERO_THRESHOLD)
        BEST_VAL_WAPE = _iterative_validation_wape(
            model_zero,
            model_qty,
            BEST_ZERO_ITER,
            BEST_QTY_ITER,
            "manual",
            threshold=BEST_THRESHOLD,
        )
        early_stop_rows.append({
            "model": "manual",
            "checkpoint": 0,
            "zero_iter": BEST_ZERO_ITER,
            "qty_iter": BEST_QTY_ITER,
            "threshold": BEST_THRESHOLD,
            "blend_alpha": 0.0,
            "val_wape": BEST_VAL_WAPE,
        })

    BEST_BLEND_ALPHA = 0.0
    mlflow.log_param("best_zero_iter_iterative", BEST_ZERO_ITER)
    mlflow.log_param("best_qty_iter_iterative", BEST_QTY_ITER)
    mlflow.log_param("best_threshold_iterative", BEST_THRESHOLD)
    mlflow.log_metric("best_val_wape_iterative_pre_blend", BEST_VAL_WAPE)
    mlflow.log_param("zero_total_iterations_final", model_zero.current_iteration())
    mlflow.log_param("qty_total_iterations_final", model_qty.current_iteration())

    if LOG_MLFLOW_MODELS:
        X_sig = X_tr.head(100).copy()
        mlflow.lightgbm.log_model(
            model_zero,
            name="iter_zero_classifier",
            signature=infer_signature(X_sig, model_zero.predict(X_sig, num_iteration=BEST_ZERO_ITER)),
        )

        X_q_sig = X_tr_nz.head(100).copy()
        mlflow.lightgbm.log_model(
            model_qty,
            name="iter_qty_regressor",
            signature=infer_signature(X_q_sig, model_qty.predict(X_q_sig, num_iteration=BEST_QTY_ITER)),
        )

print(
    f"Iterative early-stop best before blend: zi={BEST_ZERO_ITER}, qi={BEST_QTY_ITER}, "
    f"threshold={BEST_THRESHOLD:.2f}, val_wape={BEST_VAL_WAPE:.5f}"
)


## 6. Iterative Early-Stopping and Blend Summary

`BEST_ZERO_ITER` and `BEST_QTY_ITER` are selected by separate training loops. The zero model is early-stopped with a fixed warm-up quantity model; the quantity model is then early-stopped with the chosen zero model. All validation WAPE calls remain recursive and blind to true validation lags, with the threshold fixed at `0.55`. After that, a small blend against a seasonal/recent-history baseline is optionally tuned on the same blind iterative validation.


In [ ]:
early_stop_history_df = pd.DataFrame(early_stop_rows).sort_values(["model", "checkpoint"])
display(early_stop_history_df.head(MAX_DISPLAY_ROWS))

blend_rows = []
if RUN_BLEND_TUNING:
    for threshold in ZERO_THRESHOLD_CANDIDATES:
        for alpha in PREDICTION_BLEND_ALPHA_GRID:
            pred_val = score_horizon_iterative_fast(
                base_state_val,
                val_horizon,
                BEST_ZERO_ITER,
                BEST_QTY_ITER,
                float(threshold),
                f"validation_blend_thr_{threshold}_alpha_{alpha}",
                blend_alpha=float(alpha),
            )
            val_wape = wape_numpy(pred_val["quantite"].values, pred_val["prediction"].values)
            del pred_val
            gc.collect()
            blend_rows.append({"threshold": float(threshold), "blend_alpha": float(alpha), "val_wape": val_wape})
            print(f"threshold={threshold:.2f} blend alpha={alpha:.2f} -> iterative val WAPE={val_wape:.5f}")
else:
    blend_rows.append({"threshold": BEST_THRESHOLD, "blend_alpha": 0.0, "val_wape": BEST_VAL_WAPE})

blend_df = pd.DataFrame(blend_rows).sort_values("val_wape")
display(blend_df.head(MAX_DISPLAY_ROWS))
blend_best = blend_df.iloc[0]
BEST_BLEND_ALPHA = float(blend_best["blend_alpha"])
BEST_THRESHOLD_PRE_BLEND = float(BEST_THRESHOLD)
BEST_VAL_WAPE_PRE_BLEND = float(BEST_VAL_WAPE)
BEST_THRESHOLD = float(blend_best["threshold"])
BEST_VAL_WAPE = float(blend_best["val_wape"])

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("best_threshold_after_blend", BEST_THRESHOLD)
    mlflow.log_param("best_blend_alpha", BEST_BLEND_ALPHA)
    mlflow.log_metric("best_val_wape_iterative", BEST_VAL_WAPE)

print(
    f"Using iterative early-stopped model: zero_iter={BEST_ZERO_ITER}, "
    f"qty_iter={BEST_QTY_ITER}, threshold={BEST_THRESHOLD:.2f}, "
    f"blend_alpha={BEST_BLEND_ALPHA:.2f}, validation WAPE={BEST_VAL_WAPE:.5f} "
    f"(pre-blend threshold={BEST_THRESHOLD_PRE_BLEND:.2f}, WAPE={BEST_VAL_WAPE_PRE_BLEND:.5f})"
)


## 7. Iterative Validation and Internal Test


In [ ]:
val_iter = score_horizon_iterative_fast(
    base_state_val,
    val_horizon,
    BEST_ZERO_ITER,
    BEST_QTY_ITER,
    BEST_THRESHOLD,
    "validation",
    blend_alpha=BEST_BLEND_ALPHA,
)
val_wape = wape_numpy(val_iter["quantite"].values, val_iter["prediction"].values)
print(f"Validation iterative WAPE: {val_wape:.5f}")

internal_test_iter = score_horizon_iterative_fast(
    base_state_test,
    internal_test_horizon,
    BEST_ZERO_ITER,
    BEST_QTY_ITER,
    BEST_THRESHOLD,
    "internal_test",
    blend_alpha=BEST_BLEND_ALPHA,
)
internal_test_wape = wape_numpy(internal_test_iter["quantite"].values, internal_test_iter["prediction"].values)
print(f"Internal test iterative WAPE: {internal_test_wape:.5f}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_metric("val_wape_iterative_final", val_wape)
    mlflow.log_metric("internal_test_wape_iterative", internal_test_wape)

display(pd.DataFrame([
    {"split": "validation", "wape": val_wape, "blend_alpha": BEST_BLEND_ALPHA},
    {"split": "internal_test", "wape": internal_test_wape, "blend_alpha": BEST_BLEND_ALPHA},
]))


In [ ]:
def write_scored(df, table_name):
    out = df[["semaine", "code_agence", "code_article", "quantite", "p_zero", "qty_pred", "prediction", "prediction_int"]].copy()
    sdf = (
        spark.createDataFrame(out)
        .withColumn("code_agence", F.col("code_agence").cast(LongType()))
        .withColumn("code_article", F.col("code_article").cast(LongType()))
        .withColumn("prediction_int", F.col("prediction_int").cast(LongType()))
    )
    (
        sdf.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"Wrote {sdf.count():,} rows to {table_name}")

write_scored(val_iter, OUT_VAL_ITER)
write_scored(internal_test_iter, OUT_INTERNAL_TEST_ITER)


## 8. Final Iterative Leaderboard-Horizon Inference


In [ ]:
final_iter = score_horizon_iterative_fast(
    base_state_final,
    final_horizon,
    BEST_ZERO_ITER,
    BEST_QTY_ITER,
    BEST_THRESHOLD,
    "final_inference",
    blend_alpha=BEST_BLEND_ALPHA,
)
submission = final_iter[["semaine", "code_agence", "code_article", "prediction_int"]].copy()
submission = submission.rename(columns={"prediction_int": "quantite"})
submission["quantite"] = np.clip(submission["quantite"], 0, None).astype(np.int64)

submission_sdf = (
    spark.createDataFrame(submission)
    .withColumn("code_agence", F.col("code_agence").cast(LongType()))
    .withColumn("code_article", F.col("code_article").cast(LongType()))
    .withColumn("quantite", F.col("quantite").cast(LongType()))
)
(
    submission_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(OUT_FINAL_ITER)
)

n_rows = submission_sdf.count()
n_pos = submission_sdf.filter(F.col("quantite") > 0).count()
print(f"Wrote iterative final submission candidate: {OUT_FINAL_ITER}")
print(f"Rows: {n_rows:,}; positive predictions: {n_pos:,}")

with mlflow.start_run(run_id=RUN_ID):
    mlflow.log_param("iterative_final_table", OUT_FINAL_ITER)
    mlflow.log_param("iterative_final_blend_alpha", BEST_BLEND_ALPHA)
    mlflow.log_metric("iterative_final_rows", n_rows)
    mlflow.log_metric("iterative_final_positive_predictions", n_pos)
